### **Table of Content**
- Chapter 6.1: 
- Chapter 6.2: 
- Chapter 6.3: 

### **Key Highlights**
-
-
-

In [9]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv  # Import the specific functions from dotenv
from openai import OpenAI, RateLimitError, InternalServerError, APIConnectionError, APITimeoutError  # Import the specific error & OpenAI
import requests
import json
from sklearn.manifold import TSNE
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# find_dotenv() will automatically climb up from 'developer_path' 
# to the root folder to find your .env file flawlessly.
load_dotenv(find_dotenv())

# Retrieve the key
api_key = os.getenv("OPENAI_API_KEY")

# Safety check to make sure it loaded
if not api_key:
    raise ValueError("API Key is still missing! Double-check the variable name inside your .env file.")

# Initialize the client
client = OpenAI(api_key=api_key)
print("Connected successfully! OpenAI client is ready.")

Connected successfully! OpenAI client is ready.


#### **6.1 Embeddings**
- Concept from NLP, changing text into numerical format
- These texts are mapped to multi-dimensional vector spaces
- The transformed numerical form by the model are text's location in space
- Similar words appear/located closer together

Importance:
- Allow semantic (context & intention behind text) meaning to be captured
    - Semantic Search Engine
    - Recommendation Systems
    - Classification
- OpenAI: creating Embeddings endpoint 


In [6]:
# Creating a function for the prompt
def get_embedding(input): # param to pass user input/instructions
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=input
    )
    response_dict = response.model_dump() # returns a dictionary
    extracted_emdedding = response["data"][0]["embedding"] # returns a list
    return response_dict, extracted_emdedding


#### **6.2 Investigating the Vector Space**

In [4]:
articles = [
    {"headline":"Economic growth within SEA", "topic":"Business"},
    {"headline":"AI Technology advancements", "topic":"Technology"},
    {"headline":"Global climate change", "topic":"Environment"},
    {"headline":"Social media trends", "topic":"Society"},
    {"headline":"Healthcare advancements", "topic":"Healthcare"},
]

In [ ]:
# extract headline in a list
headline_text = [article["headline"] for article in articles]

get_embedding(headline_text)

In [ ]:
# embedding multiples input
for i, article in enumerate(articles):
    articles['embedding'] = response_dict['data'][i]['embedding']


Dimensionality reduction & t-SNE (t-distributed Stochastic Neighbor Embedding)

In [ ]:
emdeddings = [article['embedding'] for article in articles]
tsne = TSNE(n_components=2, perplexity=3, random_state=42)
emdeddings_2d = tsne.fit_transform(np.array(emdeddings))


Visualizing the embeddings

In [ ]:
plt.scatter(emdeddings_2d[:, 0], emdeddings_2d[:, 1])

topics = [article["topic"] for article in articles]
for i, topic in enumerate(topics):
    plt.annotate(topic, (emdeddings_2d[i, 0], emdeddings_2d[i, 1]))
plt.show()

Text Similarity
- Semantically similar text are embedded more closely in vector space
- The distance allows us to measure similarity

Measuring Similarity
- Cosine distance (the smaller, the closer it is)

In [ ]:
# creating the embeddings for the articles and the current article
def create_embeddings(texts):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    response_dict = response.model_dump()
    return [data["embedding"] for data in response_dict["data"]]

Comparing headline

In [ ]:
from scipy.spatial import distance
import numpy as np

search_text = "computer science"
search_embedding = create_embeddings([search_text])[0]

distance = []
for article in articles:
    dist = distance.cosine(search_embedding, article["embedding"])
    distance.append(dist)

min_dist_ind = np.argmin(distance)

print(f"Closest article to '{search_text}':\n{articles[min_dist_ind]['headline']}")

#### **6.3 Semantic Search & Enriched Embeddings**
- Semantic Search
    - Use embeddings to return most similar results to a search query
    - Compute cosine distances
    - Extract the texts with the smallest cosine distance

In [ ]:
articles = [
    {"headline":"Economic growth within SEA", "topic":"Business", "keywords":["economic growth", "SEA", "business"]},
    {"headline":"AI Technology advancements", "topic":"Technology", "keywords":["AI", "technology", "advancements"]},
    {"headline":"Global climate change", "topic":"Environment", "keywords":["global", "climate", "change"]},
    {"headline":"Social media trends", "topic":"Society", "keywords":["social", "media", "trends"]},
    {"headline":"Healthcare advancements", "topic":"Healthcare", "keywords":["healthcare", "advancements"]},
]

In [ ]:
def create_article_text(article):
    return f"Headline: {article['headline']}\nTopic: {article['topic']} Keywords: {", ".join(article['keywords'])}"

# creating enriched embeddings
article_texts = [create_article_text(article) for article in articles]
article_embeddings = create_embeddings(article_texts) # change text data into numerical embeddings

In [ ]:
from scipy.spatial import distance
import numpy as np

def find_n_closest(query_vector, embeddings, n=3):
    distances = []
    for index, embedding in enumerate(embeddings):
        dist = distance.cosine(query_vector, embedding)
        distances.append({"distance": dist, "index": index})
    distances_sorted = sorted(distances, key=lambda x: x["distance"])
    return distances_sorted[0:n]


query_text = "AI and technology advancements"
query_vector = create_embeddings([query_text])[0]
hits = find_n_closest(query_vector, article_embeddings, n=3)

for hit in hits:
    article = articles[hit["index"]]
    print(f"Headline: {article['headline']}, Topic: {article['topic']}, Keywords: {', '.join(article['keywords'])}, Distance: {hit['distance']}")



#### **6.3 Recommendation Systems with embeddings**
- Similar to semantic search
- Process:
    1. Embed the potential recommendations & data point
    2. Calculate cosine distances
    3. Recommend closest items


In [ ]:
# recommended articles
articles = [
    {"headline":"Economic growth within SEA", "topic":"Business", "keywords":["economic growth", "SEA", "business"]},
    {"headline":"AI Technology advancements", "topic":"Technology", "keywords":["AI", "technology", "advancements"]},
    {"headline":"Global climate change", "topic":"Environment", "keywords":["global", "climate", "change"]},
    {"headline":"Social media trends", "topic":"Society", "keywords":["social", "media", "trends"]},
    {"headline":"Healthcare advancements", "topic":"Healthcare", "keywords":["healthcare", "advancements"]},
]

# input article
current_article = {"headline":"AI and technology advancements", "topic":"Technology", "keywords":["AI", "technology", "advancements"]}

In [ ]:
## STEP 1: Create embeddings
# put the current_article into a single string
create_article_text(current_article)

# combine the features 
article_texts = [create_article_text(article) for article in articles]
current_article_text = create_article_text(current_article)

# create embeddings for the articles and the current article
current_article_embeddings = create_embeddings([current_article_text])[0]
article_embeddings = create_embeddings(article_texts)

## STEP 2: Find the closest articles
# find the closest articles
hits = find_n_closest(current_article_embeddings, article_embeddings, n=3)

## STEP 3: Print the recommended articles
# print the recommended articles
for hit in hits:
    article = articles[hit["index"]]
    print(f"Headline: {article['headline']}, Topic: {article['topic']}, Keywords: {', '.join(article['keywords'])}, Distance: {hit['distance']}")

Recommendation on multiple data points 
- Combine multiple vectors into one by taking the mean
- Compute cosine distances
- Recommend the closest vector

In [ ]:
multiple_texts = [create_article_text(article) for article in multiple_articles]
multiple_embeddings = create_embeddings(multiple_texts)
mean_history_embedding = np.mean(multiple_embeddings, axis=0)

text_filtered = [article for article in articles if article not in multiple_articles]
combine_text = [create_article_text(article) for article in text_filtered]
multi_embeddings = create_embeddings(combine_text)

hits = find_n_closest(mean_history_embedding, multi_embeddings, n=3)
for hit in hits:
    article = text_filtered[hit["index"]]
    print(f"Headline: {article['headline']}, Topic: {article['topic']}, Keywords: {', '.join(article['keywords'])}, Distance: {hit['distance']}")



#### **6.4 Embedding for Classification Tasks**
- Assigning labels to items (capture semantic means):
    - Categorization
    - Sentiment Analysis

- Classification with embeddings
    - Zero-shot classification (not using labeled data):
        1. Embed class descriptions
        2. Embed the items to classify
        3. Compute cosine distances
        4. Assign the most similar label


![alt text](<Screenshot 2026-07-23 100143.png>)
    

In [ ]:
# to classify the article into labels
article = {
    "headline": "AI and technology advancements",
    "keywords": ["AI", "technology", "advancements"]
}

topics = [
    {'label':'Tech', "description":"Articles related to technology and AI advancements."},
    {'label':'Business', "description":"Articles related to business and economics."},
    {'label':'Environment', "description":"Articles related to environmental issues."},
    {'label':'Healthcare', "description":"Articles related to healthcare and medical advancements."},
    {'label':'Society', "description":"Articles related to social issues and community development."}
]

# extract label into a single list
labels = [topic['label'] for topic in topics]

# extract label description into a single list
label_descriptions = [topic['description'] for topic in topics]
class_embeddings = create_embeddings(label_descriptions)

# create embeddings for labels
label_embeddings = create_embeddings(labels)

# create embeddings for the article
article_text = create_article_text(article)
article_embeddings = create_embeddings([article_text])[0]

# find the closest label
def find_closest(query_vector, embeddings):
    distances = []
    for index, embedding in enumerate(embeddings):
        dist = distance.cosine(query_vector, embedding)
        distances.append({"distance": dist, "index": index})
    return min(distances, key=lambda x: x["distance"])

closest_label = find_closest(article_embeddings, label_embeddings)
label = topics[closest_label["index"]]["label"]

#### **6.5 Vector Databases for Embedding Systems**
Limitation of current approach (small dataset):
    - Loading all embeddings into memory
    - Recalculate embeddings for each new query
    - Calcualting cosine distances for every embedding is slow 

Vector Database is the solution

![alt text](<Screenshot 2026-07-23 100517.png>)

Vector Database store
    - Embedding
    - Text
    - Metada data


**Creating a vector databases with ChromaDB**

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
import os
import tiktoken

client = chromadb.PersistentClient(path="./chroma_db")

# need to create a collection to store the embeddings
collection = client.create_collection(
    name="article_embeddings", # name of the collection
    embedding_function=OpenAIEmbeddingFunction(
        model_name="text-embedding-3-small",
        api_key=os.getenv("OPENAI_API_KEY")
    )
    )

client.list_collections() # inspecting all collections in the database
collection.count() # count the number of documents in the collection
collection.peek() # peek into the collection to see the first ten documents
collection.get(ids=["my-doc"]) # retrieve a document by its id

# checking token cost used
# pip install tiktoken
cost_per_1k_tokens = 0.00002
encoding = tiktoken.encoding_for_model("text-embedding-3-small")
total_tokens = sum(len(encoding.encode(text)) for text in documents)
tokens = encoding.encode("This is the source text of the document.")
cost = cost_per_1k_tokens * len(total_tokens) / 1000
print(f"Cost: ${cost}")


# insert single article into the collection
collection.add(
    ids=["my-doc"], # provide the id param
    documents=["This is the source text of the document."]
)


# insert multiple articles into the collection
collection.add(
    ids=["my-doc-1","my-doc-2"], # provide the id param
    documents=["This is the source text of the first document.", "This is the source text of the second document."]
)

**Querying and Updating the Database**

![alt text](<Screenshot 2026-07-23 134715.png>)

Retrieve the collection

In [ ]:
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

collection = client.get_collection(
    name="netflix_titles",
    embedding_function=OpenAIEmbeddingFunction( # must be specify the same embedding function used when ading data to collection
        model_name="text-embedding-3-small",
        api_key=os.getenv("OPENAI_API_KEY")
    )

)

# query the collection
result = collection.query(
    query_texts=["This is the source text of the document."],
    n_results=3 # number of results to return
)

# update collection
collection.update(
    ids=["my-doc-1","my-doc-2"], # provide the id param
    documents=["This is the updated source text of the document.", "This is the updated source text of the second document."],
    # for multiple documents, you can also use a list of dictionaries to update the collection
    # ids=[doc['id'] for doc in new_data], 
    # documents=[doc['document'] for doc in new_data]
)

collection.upsert() # upsert is a combination of update and insert. If the document exists, it will be updated. If it does not exist, it will be inserted.
collection.delete(ids=["my-doc-1","my-doc-2"]) # delete a document
client.reset() # reset the database and delete all collections

**Multiple Queries & Filtering**

In [ ]:
import csv

references_ids = [
    "s8170",
    "s8103"
]

references_texts = collection.get(ids=references_ids)["documents"]

results = collection.query(
    query_texts=references_texts,
    n_results=3
)

# adding metadata to the collection
# data from csv file
ids = []
metadatas = []

with open("netflix_titles.csv") as csvfile:
    reader = csv.DictReader(csvfile)
    for i, row in enumerate(reader):
        ids.append(row["show_id"])
        metadatas.append({
            "title": row["title"],
            "director": row["director"],
            "cast": row["cast"],
            "country": row["country"],
            "date_added": row["date_added"],
            "release_year": row["release_year"],
            "rating": row["rating"],
            "duration": row["duration"],
            "listed_in": row["listed_in"],
            "description": row["description"],
            "type": row["type"]
        })

# update the collection with metadata
collection.update(
    ids=ids,
    metadatas=metadatas
)

# query the collection
result = collection.query(
    query_texts=references_texts,
    n_results=3, # number of results to return
    where={
        "type":"Movie"     ## alternative but equivalent
    }                      # where={"type":{"$eq":"Movie"}} 
                           ## combine multiple conditions
)                          # where={"$and":[{"type":{"$eq":"Movie"}}, {"country":{"$eq":"United States"}}]}

